In [1]:
import torch
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

/home/thomas/git/bracis/slm_stability_cl/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen3.5-0.8B"

model = Qwen3_5ForCausalLM.from_pretrained(
    model_name, 
    torch_dtype="auto"
)
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 5781.14it/s]


In [8]:
GENERATION_CONFIGS = {
    "thinking": {
        "do_sample": True,
        "temperature": 0.6,
        "top_p": 0.95,
        "top_k": 20,
        "min_p": 0.0,
        "max_new_tokens": 2048,
    },
    "non_thinking": {
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "min_p": 0.0,
        "max_new_tokens": 1024,
    },
}

In [9]:
def format_prompt(
    question: str,
    task_type: str = "general",
) -> str:
    question = question.strip()

    if task_type == "math":
        return (
            f"{question}\n\n"
            "Please reason step by step, and put your final answer within \\boxed{}."
        )

    if task_type == "multiple_choice":
        return (
            f"{question}\n\n"
            "Please show your choice in the answer field with only the choice letter, "
            'e.g., "answer": "C".'
        )

    if task_type == "general":
        return question

    raise ValueError(f"Unsupported task_type: {task_type}")

In [13]:
question = "Choose an answer for the following question and give your reasons.\n\nQuestion:\nWhich figure of speech is used in this text?\nLuke's room is as tidy as an overgrown garden.\n\nChoices:\nA. verbal irony\nB. pun\n\nAnswer:"
task_type = "multiple_choice"

prompt = format_prompt(
    question=question,
    task_type=task_type
)

messages = [
    {"role": "user", "content": prompt},
]

thinking = False
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=thinking,
)

print(text)

<|im_start|>user
Choose an answer for the following question and give your reasons.

Question:
Which figure of speech is used in this text?
Luke's room is as tidy as an overgrown garden.

Choices:
A. verbal irony
B. pun

Answer:

Please show your choice in the answer field with only the choice letter, e.g., "answer": "C".<|im_end|>
<|im_start|>assistant
<think>

</think>




In [14]:
inputs = tokenizer(
    text,
    return_tensors="pt",
).to(device)

mode = "thinking" if thinking else "non_thinking"

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        **GENERATION_CONFIGS[mode],
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True,
)

In [15]:
print(response.strip())

The phrase "overgrown garden" is a metaphor describing a room that is messy, cluttered, and unorganized, while the word "tidy" suggests cleanliness and order. This creates a direct contradiction between the description of the room and the word used to describe it. The speaker is making a point by stating that their room is not tidy, yet they compare it to a garden that is overgrown. This contrast between the literal meaning of "tidy" and the figurative meaning of "overgrown" is known as **verbal irony**.

Answer: A
